# Bus-Branch Transmission Modeling

Bus-branch models are the planning view of a transmission network (as in PSS/E or MATPOWER cases): each bus is a single node, and the substation switching detail is not represented. In CIM, a bus is a `ConnectivityNode`, branches are `ACLineSegment` and `PowerTransformer` objects, and loads, generators and shunts connect to the buses through their `Terminal` objects. The CIMantic Graphs `BusBranchModel` class gives access to these models.

## Creating a BusBranchModel

This page uses the IEEE 118 bus test model, read with the `XMLFile` parser. Import the CIM profile, set the matching environment variable and the `BusBranchModel` class.

The test file also contains dynamics classes (e.g. `ExcitationSystem`) that are not in the `cim17v40` profile. Setting `CIMG_VALIDATION_LOG_LEVEL` to `DEBUG` logs the messages about them at debug level, so they are not printed:

In [1]:
import os
os.environ['CIMG_CIM_PROFILE'] = 'cim17v40'  # import and env var must match
os.environ['CIMG_VALIDATION_LOG_LEVEL'] = 'DEBUG'
import cimprofiles.cim17.cim17v40 as cim

from cimgraph.databases import XMLFile
from cimgraph.models import BusBranchModel

Create the CIM `ConnectivityNodeContainer` that holds the model, and load the model:

In [2]:
file = XMLFile(filename='../../tests/test_models/IEEE118_CIM.xml')

model_mrid = '1783D2A8-1204-4781-A0B4-7A73A2FA6038'  # IEEE 118 Bus
container = cim.ConnectivityNodeContainer(mRID=model_mrid)
network = BusBranchModel(connection=file, container=container)

In [3]:
for cim_class in [cim.ConnectivityNode, cim.ACLineSegment, cim.PowerTransformer,
                  cim.EnergyConsumer, cim.SynchronousMachine, cim.LinearShuntCompensator]:
    print(cim_class.__name__, len(network.list_by_class(cim_class)))

ConnectivityNode 193
ACLineSegment 170
PowerTransformer 84
EnergyConsumer 99
SynchronousMachine 56
LinearShuntCompensator 14


## Equipment at a Bus

Each bus lists the terminals connected to it, and each terminal its equipment:

In [4]:
bus = network.first(cim.ConnectivityNode)
print('bus', bus.name)
for terminal in bus.Terminals:
    equipment = terminal.ConductingEquipment
    print('   ', equipment.__class__.__name__, equipment.name)

bus 17
    ACLineSegment 15_17_1
    ACLineSegment 16_17_1
    ACLineSegment 17_18_1
    ACLineSegment 17_31_1
    ACLineSegment 17_113_1
    EnergyConsumer 17_1
    PowerTransformer 30_17_0_1


## Branch Impedances

Impedances are stored in ohms. To convert them to per unit, divide by the base impedance `z_base = V_base² / S_base`, using the model's system base `network.base_power` (100 MVA here) and the nominal voltage of the line:

In [5]:
base_power_va = network.base_power.to('VA')

for line in network.list_by_class(cim.ACLineSegment)[:5]:
    base_voltage_v = line.BaseVoltage.nominalVoltage.to('V')
    z_base = base_voltage_v ** 2 / base_power_va
    r_pu = line.r.to('ohm') / z_base
    x_pu = line.x.to('ohm') / z_base
    nominal_kv = line.BaseVoltage.nominalVoltage.to('kV')
    print(f'{line.name}: {nominal_kv} kV, r = {r_pu:.5f} pu, x = {x_pu:.5f} pu')

64_65_1: 345.0 kV, r = 0.00269 pu, x = 0.03020 pu
63_64_1: 345.0 kV, r = 0.00172 pu, x = 0.02000 pu
109_110_1: 138.0 kV, r = 0.02780 pu, x = 0.07620 pu
2_12_1: 138.0 kV, r = 0.01870 pu, x = 0.06160 pu
21_22_1: 138.0 kV, r = 0.02090 pu, x = 0.09700 pu


## Loads by Voltage Level

Each `BaseVoltage` lists the equipment at that voltage. Converting the load values with `.to()` gives the total load per voltage level in MW and MVAr:

In [6]:
for base_voltage in network.list_by_class(cim.BaseVoltage):
    loads = [equipment for equipment in base_voltage.ConductingEquipment
             if isinstance(equipment, cim.EnergyConsumer)]
    total_p = sum(load.p.to('MW') for load in loads)
    total_q = sum(load.q.to('MVAr') for load in loads)
    print(f'{base_voltage.name}: {len(loads)} loads, {total_p:.1f} MW, {total_q:.1f} MVAr')

BV_13.80: 0 loads, 0.0 MW, 0.0 MVAr
BV_345.00: 2 loads, 212.0 MW, 0.0 MVAr
BV_138.00: 97 loads, 4030.0 MW, 1438.0 MVAr


## Loading a BusBranchModel from a Database

With a database, pass its connection instead of the `XMLFile`. The example below needs a running Blazegraph database with the IEEE 118 model loaded and the `sparql` extra (`pip install cim-graph[sparql]`), so it is not run here. Database models are loaded with only the identifiers of the equipment, nodes and terminals; `.get_all_edges()` reads the attributes of a class:

```python
os.environ['CIMG_URL'] = 'http://localhost:8889/bigdata/namespace/kb/sparql'
from cimgraph.databases import BlazegraphConnection

database = BlazegraphConnection()
network = BusBranchModel(connection=database, container=container)
network.get_all_edges(cim.ACLineSegment)
network.pprint(cim.ACLineSegment)
```

## Distributed Modeling

`BusBranchModel(..., distributed=True)` is not implemented yet and builds an empty graph. Use the centralized model.